<a href="https://colab.research.google.com/github/Poojarautela03/ABTALKS/blob/main/Day_20_Build%20a%20Working%20AI%20Knowledge%20Assistant/AI_Assistant.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Day 20 — Build a Working AI Knowledge Assistant 🤖

A deployed, working RAG API built and run **inside this Colab notebook**. Combines Day 12 chunking, Day 14 FAISS semantic search, Day 17 metadata filtering, and Day 19's grounding system prompt into a FastAPI service, run in-notebook using `nest_asyncio` so the server and test calls can both run in the same session.

> **Provider note:** uses **Google Gemini** (`gemini-embedding-001` + `gemini-flash-lite-latest`) instead of OpenAI, due to OpenAI API quota limits hit earlier in the challenge.

In [1]:
!pip install -q fastapi "uvicorn[standard]" pydantic numpy faiss-cpu google-generativeai langchain-text-splitters nest_asyncio requests

import os, sys, json, time, threading
from datetime import datetime
from typing import List, Dict, Any, Optional

import numpy as np
import faiss
import google.generativeai as genai
from langchain_text_splitters import RecursiveCharacterTextSplitter
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
import uvicorn
import nest_asyncio
import requests

from google.colab import userdata

nest_asyncio.apply()
print("Setup done.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 40.1 MB/s eta 0:00:00


/usr/local/lib/python3.13/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


Setup done.


In [3]:
GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')
EMBEDDING_MODEL = "models/gemini-embedding-001"
CHAT_MODEL_NAME = "gemini-flash-lite-latest"
LOW_CONFIDENCE_THRESHOLD = 0.3
CHUNK_SIZE = 500
CHUNK_OVERLAP = 100

genai.configure(api_key=GEMINI_API_KEY)
chat_model = genai.GenerativeModel(CHAT_MODEL_NAME)

GROUNDING_SYSTEM_PROMPT = """You are a strict, grounded knowledge assistant. You must ONLY
answer using information explicitly stated in the provided context below.

Rules:
1. If the answer is not directly supported by the context, respond exactly with:
   "I cannot answer this based on the available knowledge base."
2. Do NOT use any external knowledge, training data, or assumptions to fill gaps.
3. Do NOT guess, infer, or extrapolate beyond what is explicitly written in the context.
4. Cite which source(s) you used by referencing their [source_id] tags inline.

CONTEXT:
{context}

QUESTION:
{question}

ANSWER (include [source_id] citations inline where you use a fact):"""

print("Config loaded.")

Config loaded.


## Document store: raw documents with metadata (Day 17 pattern)

In [4]:
class Document:
    def __init__(self, doc_id, title, text, category, date, document_type):
        self.doc_id = doc_id
        self.title = title
        self.text = text
        self.category = category
        self.date = date
        self.document_type = document_type


class Chunk:
    def __init__(self, chunk_id, text, source_doc: Document):
        self.chunk_id = chunk_id
        self.text = text
        self.source_doc = source_doc

    def to_dict(self) -> Dict[str, Any]:
        return {
            "chunk_id": self.chunk_id,
            "text": self.text,
            "source_id": self.source_doc.doc_id,
            "source_title": self.source_doc.title,
            "category": self.source_doc.category,
            "date": self.source_doc.date,
            "document_type": self.source_doc.document_type,
        }

print("Document/Chunk classes defined.")

Document/Chunk classes defined.


## Knowledge base — Nimbus Robotics sample documents

In [5]:
DOCUMENTS = [
    Document(
        doc_id="doc_company_history", title="Company History",
        text=(
            "Nimbus Robotics was founded in 2031 by engineer Priya Kalathil in Pune, India. "
            "The company began as a three-person team working out of a small office near "
            "Hinjewadi, focused on solving warehouse automation problems for mid-sized logistics "
            "firms. Within its first two years, Nimbus Robotics grew to employ 212 people across "
            "three offices: Pune, Bengaluru, and Singapore. The company's early funding came from "
            "a mix of angel investors and a seed round led by a regional venture fund focused on "
            "industrial robotics. Priya Kalathil remains CEO as of 2034, and has spoken publicly "
            "about her vision of making warehouse automation affordable for small and mid-size "
            "businesses, not just large logistics conglomerates."
        ),
        category="company_background", date="2031-03-15", document_type="reference",
    ),
    Document(
        doc_id="doc_leadership", title="Leadership Team",
        text=(
            "The company's CTO, Rohan Mehta, previously led robotics research at a university lab "
            "for eight years before joining Nimbus Robotics in 2031 as a founding engineer. Rohan "
            "holds a doctorate in mechanical engineering and has published research on adaptive "
            "gripping mechanisms, which directly informed the design of the Aster-7's FlexGrip "
            "system. The leadership team also includes Ananya Desai as VP of Operations, who joined "
            "in 2032 after a decade in supply chain management at a multinational logistics firm."
        ),
        category="company_background", date="2031-04-01", document_type="reference",
    ),
    Document(
        doc_id="doc_aster7_specs", title="Aster-7 Product Specifications",
        text=(
            "Nimbus Robotics' flagship product is the Aster-7, a warehouse picking robot with a "
            "99.2% accuracy rate. The Aster-7 uses a proprietary gripper called FlexGrip, which "
            "adjusts pressure using 12 micro-sensors per finger, allowing it to handle items ranging "
            "from fragile glassware to heavy machine parts without reconfiguration. The Aster-7's "
            "battery lasts 14 hours on a single charge and recharges fully in 40 minutes, making it "
            "suitable for round-the-clock warehouse operations with staggered charging cycles. The "
            "robot's navigation system relies on LIDAR combined with a pre-mapped warehouse layout, "
            "updated dynamically as shelving changes."
        ),
        category="product", date="2033-01-10", document_type="technical",
    ),
    Document(
        doc_id="doc_aster7_deployment", title="Aster-7 Market Deployment",
        text=(
            "The Aster-7 has been deployed in over 60 warehouses across South and Southeast Asia "
            "since its commercial launch in 2033. Early customers include regional e-commerce "
            "fulfillment centers and a handful of pharmaceutical distribution warehouses, where the "
            "Aster-7's precision handling was particularly valued for reducing breakage rates on "
            "fragile packaging. Customer feedback collected through 2033 and 2034 has been generally "
            "positive, with the most common request being for improved battery swap mechanisms to "
            "reduce downtime during peak shipping seasons."
        ),
        category="market", date="2033-08-15", document_type="marketing",
    ),
    Document(
        doc_id="doc_aster8_roadmap", title="Aster-8 Product Roadmap",
        text=(
            "Nimbus Robotics' next product, the Aster-8, is scheduled for release in early 2035. "
            "The Aster-8 is designed to address the most common customer request from Aster-7 "
            "deployments: faster battery swapping. Early engineering previews suggest the Aster-8 "
            "will support hot-swappable battery packs, reducing downtime from 40 minutes to under "
            "5 minutes. The Aster-8 is also expected to include an upgraded version of FlexGrip with "
            "18 micro-sensors per finger, improving handling precision for smaller items."
        ),
        category="product", date="2034-09-01", document_type="marketing",
    ),
    Document(
        doc_id="doc_finance", title="Financial Performance",
        text=(
            "Nimbus Robotics reported revenue of 340 million rupees in fiscal year 2033, an increase "
            "of approximately 65% over the prior year, driven primarily by expanded Aster-7 "
            "deployments in Southeast Asia. The company has not yet reached profitability, with "
            "continued investment in R&D for the Aster-8 program cited as the primary driver of "
            "ongoing operating losses. Nimbus Robotics closed a Series B funding round in late 2033 "
            "to support this continued product development."
        ),
        category="finance", date="2033-12-31", document_type="reference",
    ),
    Document(
        doc_id="doc_competition", title="Competitive Landscape",
        text=(
            "Nimbus Robotics' main competitor is Solace Automation, founded a year earlier in 2030. "
            "Solace Automation focuses on a broader range of warehouse robotics, including both "
            "picking robots and autonomous forklifts, giving it a wider product portfolio than "
            "Nimbus Robotics' single-product focus on the Aster line. Industry analysts have noted "
            "that Nimbus Robotics' narrower focus has allowed it to iterate faster on gripper "
            "precision, which has become its primary competitive differentiator in the pharmaceutical "
            "and fragile-goods segments of the market."
        ),
        category="market", date="2032-06-01", document_type="marketing",
    ),
]

print(f"Loaded {len(DOCUMENTS)} documents.")

Loaded 7 documents.


## Chunking (Day 12)

In [6]:
def chunk_documents(documents, chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP):
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size, chunk_overlap=chunk_overlap,
        separators=["\n\n", "\n", ". ", " ", ""],
    )
    chunks = []
    for doc in documents:
        pieces = splitter.split_text(doc.text)
        for i, piece in enumerate(pieces):
            chunk_id = f"{doc.doc_id}::chunk{i}"
            chunks.append(Chunk(chunk_id=chunk_id, text=piece, source_doc=doc))
    return chunks

_chunks_preview = chunk_documents(DOCUMENTS)
print(f"Produced {len(_chunks_preview)} chunks from {len(DOCUMENTS)} documents.")

Produced 12 chunks from 7 documents.


## Embeddings + FAISS index (Day 14) + metadata filtering (Day 17)

In [7]:
def get_embeddings(texts, max_retries=3):
    embeddings = []
    for text in texts:
        for attempt in range(max_retries):
            try:
                result = genai.embed_content(model=EMBEDDING_MODEL, content=text)
                embeddings.append(result["embedding"])
                break
            except Exception as e:
                if attempt < max_retries - 1:
                    time.sleep(5)
                else:
                    raise RuntimeError(f"Embedding failed after {max_retries} attempts: {e}")
        time.sleep(1)
    return embeddings


class RAGIndex:
    def __init__(self, chunks):
        self.chunks = chunks
        texts = [c.text for c in chunks]
        vectors = np.array(get_embeddings(texts), dtype=np.float32)
        self.dimension = vectors.shape[1]
        self.index = faiss.IndexFlatL2(self.dimension)
        self.index.add(vectors)

    def search(self, query, top_k=3, filters=None, search_pool=10):
        query_vector = np.array(get_embeddings([query]), dtype=np.float32)
        distances, indices = self.index.search(query_vector, search_pool)

        results = []
        for dist, idx in zip(distances[0], indices[0]):
            if idx == -1:
                continue
            chunk = self.chunks[idx]

            if filters:
                if "category" in filters and chunk.source_doc.category != filters["category"]:
                    continue
                if "document_type" in filters and chunk.source_doc.document_type != filters["document_type"]:
                    continue
                if "date_after" in filters:
                    doc_date = datetime.strptime(chunk.source_doc.date, "%Y-%m-%d")
                    cutoff = datetime.strptime(filters["date_after"], "%Y-%m-%d")
                    if doc_date < cutoff:
                        continue

            entry = chunk.to_dict()
            entry["distance"] = float(dist)
            entry["similarity"] = float(1 / (1 + dist))
            results.append(entry)
            if len(results) >= top_k:
                break
        return results

print("Building the FAISS index (this embeds every chunk, may take ~30-60s)...")
_chunks = chunk_documents(DOCUMENTS)
_index = RAGIndex(_chunks)
print(f"Index built with {len(_chunks)} chunks.")

Building the FAISS index (this embeds every chunk, may take ~30-60s)...
Index built with 12 chunks.


## Grounded generation with citations + confidence (Day 19 + this task)

In [8]:
def generate_answer(query, index, top_k=3, filters=None):
    retrieved = index.search(query, top_k=top_k, filters=filters)

    if not retrieved:
        return {"answer": "I cannot answer this based on the available knowledge base.",
                "sources": [], "low_confidence": True, "top_similarity": 0.0}

    context_lines = [f"[{r['source_id']}] ({r['source_title']}): {r['text']}" for r in retrieved]
    context = "\n".join(context_lines)

    prompt = GROUNDING_SYSTEM_PROMPT.format(context=context, question=query)
    response = chat_model.generate_content(
        prompt, generation_config=genai.types.GenerationConfig(temperature=0)
    )
    answer_text = response.text.strip()

    best_similarity = max(r["similarity"] for r in retrieved)
    low_confidence = best_similarity < LOW_CONFIDENCE_THRESHOLD

    if low_confidence:
        answer_text += ("\n\n\u26a0\ufe0f Low confidence: the retrieved information may not be "
                         "well-supported for this question. Treat this answer with caution.")

    sources = [{"source_id": r["source_id"], "title": r["source_title"],
                "chunk_id": r["chunk_id"], "similarity": round(r["similarity"], 4)} for r in retrieved]

    return {"answer": answer_text, "sources": sources,
            "low_confidence": low_confidence, "top_similarity": round(best_similarity, 4)}

# Quick sanity check
test = generate_answer("Who founded Nimbus Robotics?", _index, top_k=3)
print(json.dumps(test, indent=2))

{
  "answer": "Nimbus Robotics was founded by engineer Priya Kalathil [doc_company_history].",
  "sources": [
    {
      "source_id": "doc_company_history",
      "title": "Company History",
      "chunk_id": "doc_company_history::chunk0",
      "similarity": 0.707
    },
    {
      "source_id": "doc_aster7_specs",
      "title": "Aster-7 Product Specifications",
      "chunk_id": "doc_aster7_specs::chunk0",
      "similarity": 0.6732
    },
    {
      "source_id": "doc_competition",
      "title": "Competitive Landscape",
      "chunk_id": "doc_competition::chunk0",
      "similarity": 0.6669
    }
  ],
  "low_confidence": false,
  "top_similarity": 0.707
}


## FastAPI app exposing POST /ask

In [9]:
app = FastAPI(
    title="Nimbus Knowledge Assistant",
    description="A grounded RAG-based Q&A API built for ABTalks Day 20.",
    version="1.0.0",
)

class AskRequest(BaseModel):
    query: str
    top_k: Optional[int] = 3
    filters: Optional[Dict[str, Any]] = None

class SourceItem(BaseModel):
    source_id: str
    title: str
    chunk_id: str
    similarity: float

class AskResponse(BaseModel):
    answer: str
    sources: List[SourceItem]
    low_confidence: bool
    top_similarity: float

@app.get("/")
def health_check():
    return {"status": "ok", "indexed_chunks": len(_chunks)}

@app.post("/ask", response_model=AskResponse)
def ask(request: AskRequest):
    if not request.query or not request.query.strip():
        raise HTTPException(status_code=400, detail="Field 'query' must not be empty.")
    result = generate_answer(query=request.query, index=_index,
                              top_k=request.top_k or 3, filters=request.filters)
    return result

print("FastAPI app defined.")

FastAPI app defined.


## Run the server in-notebook

Colab can't block on `uvicorn.run()` in the main thread (it would freeze the notebook), so the server runs in a background thread using `nest_asyncio`. This gives you a real local API you can hit with `requests` or `curl` from another cell — same as running it in a terminal.

In [10]:
def run_server():
    uvicorn.run(app, host="127.0.0.1", port=8000, log_level="warning")

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()
time.sleep(3)
print("Server started at http://127.0.0.1:8000")

Server started at http://127.0.0.1:8000


## Test the /ask endpoint (equivalent to curl)

In [11]:
# Health check
r = requests.get("http://127.0.0.1:8000/")
print("Health check:", r.json())

# Equivalent to:
# curl -X POST http://127.0.0.1:8000/ask -H "Content-Type: application/json" \
#   -d '{"query": "Who founded Nimbus Robotics and what is the Aster-7?"}'
r = requests.post(
    "http://127.0.0.1:8000/ask",
    json={"query": "Who founded Nimbus Robotics and what is the Aster-7?"}
)
print(json.dumps(r.json(), indent=2))

Health check: {'status': 'ok', 'indexed_chunks': 12}
{
  "answer": "I cannot answer this based on the available knowledge base.",
  "sources": [
    {
      "source_id": "doc_aster7_specs",
      "title": "Aster-7 Product Specifications",
      "chunk_id": "doc_aster7_specs::chunk0",
      "similarity": 0.7427
    },
    {
      "source_id": "doc_leadership",
      "title": "Leadership Team",
      "chunk_id": "doc_leadership::chunk0",
      "similarity": 0.6931
    },
    {
      "source_id": "doc_aster8_roadmap",
      "title": "Aster-8 Product Roadmap",
      "chunk_id": "doc_aster8_roadmap::chunk0",
      "similarity": 0.6795
    }
  ],
  "low_confidence": false,
  "top_similarity": 0.7427
}


In [12]:
# Same request, from an actual curl command run inside the notebook via bash magic
!curl -s -X POST http://127.0.0.1:8000/ask -H "Content-Type: application/json" -d '{"query": "What products does Nimbus Robotics make?", "filters": {"category": "product"}}'

{"answer":"Based on the provided context, Nimbus Robotics makes the Aster-7 (a warehouse picking robot) [doc_aster7_specs] and has a next product scheduled for release called the Aster-8 [doc_aster8_roadmap].","sources":[{"source_id":"doc_aster7_specs","title":"Aster-7 Product Specifications","chunk_id":"doc_aster7_specs::chunk0","similarity":0.6868},{"source_id":"doc_aster8_roadmap","title":"Aster-8 Product Roadmap","chunk_id":"doc_aster8_roadmap::chunk0","similarity":0.6513},{"source_id":"doc_aster7_specs","title":"Aster-7 Product Specifications","chunk_id":"doc_aster7_specs::chunk1","similarity":0.5375}],"low_confidence":false,"top_similarity":0.6868}

## 15-query test suite (5 easy, 5 medium, 5 hard)

In [13]:
TEST_QUERIES = [
    {"query": "Who founded Nimbus Robotics?", "difficulty": "easy"},
    {"query": "What is the Aster-7's battery life?", "difficulty": "easy"},
    {"query": "How much revenue did Nimbus Robotics report in fiscal year 2033?", "difficulty": "easy"},
    {"query": "Who is Nimbus Robotics' main competitor?", "difficulty": "easy"},
    {"query": "When is the Aster-8 scheduled for release?", "difficulty": "easy"},
    {"query": "What is the Aster-7's gripper called and how does it work?", "difficulty": "medium"},
    {"query": "How many people does Nimbus Robotics employ and across how many offices?", "difficulty": "medium"},
    {"query": "What background does the CTO have?", "difficulty": "medium"},
    {"query": "Where has the Aster-7 been deployed?", "difficulty": "medium"},
    {"query": "What is different about Solace Automation's product range compared to Nimbus Robotics?", "difficulty": "medium"},
    {"query": "What is the most common customer complaint and how is Nimbus addressing it in future products?", "difficulty": "hard"},
    {"query": "How has Nimbus Robotics' funding history evolved over time?", "difficulty": "hard"},
    {"query": "What color is the Aster-7 robot?", "difficulty": "hard"},
    {"query": "What is Nimbus Robotics' stock ticker symbol?", "difficulty": "hard"},
    {"query": "How does the Aster-8's sensor count compare to the Aster-7's?", "difficulty": "hard"},
]

results = []
for i, item in enumerate(TEST_QUERIES, start=1):
    print(f"[{i}/15] ({item['difficulty']}) {item['query']}")
    r = requests.post("http://127.0.0.1:8000/ask", json={"query": item["query"]})
    result = r.json()

    print(f"  Answer: {result['answer']}")
    print(f"  Sources: {[s['source_id'] for s in result['sources']]}")
    print(f"  Top similarity: {result['top_similarity']}")
    print(f"  Low confidence flag: {result['low_confidence']}\n")

    results.append({
        "query": item["query"], "difficulty": item["difficulty"],
        "answer": result["answer"], "sources": result["sources"],
        "top_similarity": result["top_similarity"], "low_confidence": result["low_confidence"],
        "retrieval_quality_1to5": None, "response_quality_1to5": None,
    })

with open("test_results.json", "w") as f:
    json.dump(results, f, indent=2)
print("Saved results to test_results.json — fill in the two quality score fields for each entry.")

[1/15] (easy) Who founded Nimbus Robotics?
  Answer: Nimbus Robotics was founded by engineer Priya Kalathil [doc_company_history].
  Sources: ['doc_company_history', 'doc_aster7_specs', 'doc_competition']
  Top similarity: 0.707
  Low confidence flag: False

[2/15] (easy) What is the Aster-7's battery life?
  Answer: The Aster-7's battery lasts 14 hours on a single charge [doc_aster7_specs].
  Sources: ['doc_aster7_specs', 'doc_aster7_deployment', 'doc_aster8_roadmap']
  Top similarity: 0.6996
  Low confidence flag: False

[3/15] (easy) How much revenue did Nimbus Robotics report in fiscal year 2033?
  Answer: Nimbus Robotics reported revenue of 340 million rupees in fiscal year 2033 [doc_finance].
  Sources: ['doc_finance', 'doc_company_history', 'doc_competition']
  Top similarity: 0.7615
  Low confidence flag: False

[4/15] (easy) Who is Nimbus Robotics' main competitor?
  Answer: Nimbus Robotics' main competitor is Solace Automation [doc_competition].
  Sources: ['doc_competition',